# Data Preprocessing

### Loading the dataset

In [2]:
import pandas as pd
import datetime as dt

df = pd.read_csv('data/export_alimconfiance.csv', delimiter=";")

### Data types

In [3]:
for col in df.columns:  
    print("Type of " + col + " : " + str(df[col].dtype))

Type of APP_Libelle_etablissement : object
Type of SIRET : object
Type of Adresse_2_UA : object
Type of Code_postal : float64
Type of Libelle_commune : object
Type of Numero_inspection : object
Type of Date_inspection : object
Type of APP_Libelle_activite_etablissement : object
Type of Synthese_eval_sanit : object
Type of APP_Code_synthese_eval_sanit : int64
Type of Agrement : object
Type of geores : object
Type of filtre : object
Type of ods_type_activite : object
Type of reg_name : object
Type of reg_code : float64
Type of dep_name : object
Type of dep_code : object
Type of com_name : object
Type of com_code : object


### Converting 'Date_inspection' into datetime

In [4]:
df['Date_inspection'] = pd.to_datetime(df['Date_inspection'], utc=True).dt.tz_localize(None)
print(df['Date_inspection'].values.dtype)

datetime64[ns]


### Searching for NaN values

In [5]:
for col in df.columns:
    print("Number of NaN values of " + col + " : " + str(df[col].isna().sum()))

Number of NaN values of APP_Libelle_etablissement : 0
Number of NaN values of SIRET : 0
Number of NaN values of Adresse_2_UA : 409
Number of NaN values of Code_postal : 26
Number of NaN values of Libelle_commune : 0
Number of NaN values of Numero_inspection : 0
Number of NaN values of Date_inspection : 0
Number of NaN values of APP_Libelle_activite_etablissement : 0
Number of NaN values of Synthese_eval_sanit : 0
Number of NaN values of APP_Code_synthese_eval_sanit : 0
Number of NaN values of Agrement : 34546
Number of NaN values of geores : 1225
Number of NaN values of filtre : 8444
Number of NaN values of ods_type_activite : 0
Number of NaN values of reg_name : 1268
Number of NaN values of reg_code : 1268
Number of NaN values of dep_name : 1268
Number of NaN values of dep_code : 1268
Number of NaN values of com_name : 1268
Number of NaN values of com_code : 1268


### Dropping columns with columns not required for the analysis and those containing many null values : 'SIRET', 'Numero_Inspection', 'Agrement'

In [6]:
df = df.drop(['SIRET', 'Numero_inspection', 'Agrement', 'ods_type_activite'], axis=1)

### Suppresion of the Nan values

In [7]:
for col in df.columns:
    df.dropna(subset=[col], inplace=True)

### Identifying unique the number of unique values for each columns

In [8]:
for col in df.columns:  
    print("Number of unique values of " + col + " : " + str(df[col].unique().size))

Number of unique values of APP_Libelle_etablissement : 27546
Number of unique values of Adresse_2_UA : 29923
Number of unique values of Code_postal : 4705
Number of unique values of Libelle_commune : 7476
Number of unique values of Date_inspection : 309
Number of unique values of APP_Libelle_activite_etablissement : 140
Number of unique values of Synthese_eval_sanit : 4
Number of unique values of APP_Code_synthese_eval_sanit : 4
Number of unique values of geores : 30581
Number of unique values of filtre : 5
Number of unique values of reg_name : 21
Number of unique values of reg_code : 21
Number of unique values of dep_name : 104
Number of unique values of dep_code : 104
Number of unique values of com_name : 7389
Number of unique values of com_code : 7491


We can see that certain establishments have been inspected more than one time, so we will have to study this by paying attention to the inspection date.

### List of the possible outcome of an inspection

In [9]:
print(df["Synthese_eval_sanit"].unique())

['Satisfaisant' 'A améliorer' 'Très satisfaisant'
 'A corriger de manière urgente']


### Changing types of columns in ordre to have : int, string, datetime or object

In [10]:
for col in df.columns:
    if col == 'Code_postal' or col == 'reg_code' or col == 'APP_Code_synthese_eval_sanit':
        df[col] = df[col].astype(int)
    else:
        if col != 'Date_inspection' and col != 'geores':
            df[col] = df[col].astype(str)

for col in df.columns:  
    print("Type of " + col + " : " + str(df[col].dtype))


Type of APP_Libelle_etablissement : object
Type of Adresse_2_UA : object
Type of Code_postal : int64
Type of Libelle_commune : object
Type of Date_inspection : datetime64[ns]
Type of APP_Libelle_activite_etablissement : object
Type of Synthese_eval_sanit : object
Type of APP_Code_synthese_eval_sanit : int64
Type of geores : object
Type of filtre : object
Type of reg_name : object
Type of reg_code : int64
Type of dep_name : object
Type of dep_code : object
Type of com_name : object
Type of com_code : object


### Some dataframe explorations

In [15]:
syntheses = df.groupby('Synthese_eval_sanit').size()
print(syntheses)

Synthese_eval_sanit
A améliorer                       1814
A corriger de manière urgente      230
Satisfaisant                     20881
Très satisfaisant                10245
dtype: int64


In [16]:
syntheses_par_region = df.groupby(['Synthese_eval_sanit', 'reg_name']).size().reset_index(name='Evaluation')
syntheses_par_region

,Synthese_eval_sanit,reg_name,Evaluation
0,A améliorer,Auvergne-Rhône-Alpes,123
1,A améliorer,Bourgogne-Franche-Comté,61
2,A améliorer,Bretagne,54
3,A améliorer,Centre-Val de Loire,132
4,A améliorer,Corse,43
...,...,...,...
69,Très satisfaisant,Occitanie,1345
70,Très satisfaisant,Pays de la Loire,836
71,Très satisfaisant,Provence-Alpes-Côte d'Azur,515
72,Très satisfaisant,Saint-Martin,15


In [17]:
syntheses_par_region_activite = df.groupby(['Synthese_eval_sanit', 'reg_name', 'APP_Libelle_activite_etablissement']).size().reset_index(name='Evaluation')
syntheses_par_region_activite

,Synthese_eval_sanit,reg_name,APP_Libelle_activite_etablissement,Evaluation
0,A améliorer,Auvergne-Rhône-Alpes,Boucherie-Charcuterie,9
1,A améliorer,Auvergne-Rhône-Alpes,Boulangerie-Pâtisserie,7
2,A améliorer,Auvergne-Rhône-Alpes,Glacier,1
3,A améliorer,Auvergne-Rhône-Alpes,Libre service,1
4,A améliorer,Auvergne-Rhône-Alpes,Libre service|Alimentation générale,1
...,...,...,...,...
1175,Très satisfaisant,Île-de-France,Rayon traiteur|Rayon boucherie-charcuterie,1
1176,Très satisfaisant,Île-de-France,Restaurants,1207
1177,Très satisfaisant,Île-de-France,Restauration collective,363
1178,Très satisfaisant,Île-de-France,Traiteur,38
